# Fire Detection & Tracking on Video (YOLO26 + ByteTrack + Supervision)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/Supervision_Video_Inferencing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Runs the fire-detection model from `drone_fire_detection_yolo26.ipynb` over a video, tracks each detection
across frames, and writes an annotated result video.

**Runtime:** `Runtime` → `Change runtime type` → **T4 GPU**. Video inference on CPU is slow.

> **What changed:** tracking used to require cloning
> [ByteTrack](https://github.com/ifzhang/ByteTrack), building YOLOX from source, and
> installing `onemetric` / `cython_bbox` — a toolchain that no longer builds on current
> Python. ByteTrack now ships inside Ultralytics, so `model.track(...)` handles it with
> no extra dependencies, and Supervision reads the track ids straight off the result.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install

In [ ]:
%pip install -q "ultralytics>=8.4.122" "supervision>=0.30.0"

import supervision as sv
import ultralytics

print("ultralytics:", ultralytics.__version__)
print("supervision:", sv.__version__)

## 3. Weights and source video

`best.pt` is not committed to the repo — upload the checkpoint you exported from
`drone_fire_detection_yolo26.ipynb`. The sample `fire.mp4` is pulled from the repo automatically.

In [ ]:
from pathlib import Path

MODEL_PATH = Path("best.pt")
SOURCE_VIDEO_PATH = Path("fire.mp4")
TARGET_VIDEO_PATH = Path("fire_result.mp4")

if not SOURCE_VIDEO_PATH.exists():
    !wget -q -O {SOURCE_VIDEO_PATH} https://github.com/jakkzz/Fire-Detection-Drone/raw/main/fire.mp4

if not MODEL_PATH.exists():
    print("best.pt not found — upload it now (or run the training notebook first).")
    try:
        from google.colab import files  # type: ignore

        files.upload()
    except ImportError:
        raise FileNotFoundError("Place best.pt next to this notebook.")

video_info = sv.VideoInfo.from_video_path(str(SOURCE_VIDEO_PATH))
print(video_info)

## 4. Load the model

In [ ]:
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
model.fuse()

print("classes:", model.names)

## 5. Annotate a single frame first

Cheap sanity check before committing to a full-video pass.

In [ ]:
import cv2

CONFIDENCE_THRESHOLD = 0.25

box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.6, text_thickness=2, text_padding=5)


def make_labels(detections: sv.Detections) -> list[str]:
    """Build `#id class conf` labels, omitting the id when tracking is off."""
    labels = []
    for i in range(len(detections)):
        name = detections["class_name"][i]
        conf = detections.confidence[i]
        tracker_id = None if detections.tracker_id is None else detections.tracker_id[i]
        prefix = "" if tracker_id is None else f"#{tracker_id} "
        labels.append(f"{prefix}{name} {conf:.2f}")
    return labels


frame = next(sv.get_video_frames_generator(str(SOURCE_VIDEO_PATH)))

result = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)

annotated = box_annotator.annotate(scene=frame.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=make_labels(detections))

sv.plot_image(image=cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB), size=(10, 10))

## 6. Detect + track across the whole video

`model.track(..., persist=True, tracker="bytetrack.yaml")` keeps tracker state
between calls, so each fire region gets a stable id across frames.
`sv.Detections.from_ultralytics` reads those ids into `detections.tracker_id`,
and `TraceAnnotator` uses them to draw motion trails.

ByteTrack is designed to consume *low*-confidence detections — it recovers weak
boxes by matching them against existing tracks, which is where much of its
accuracy comes from. So `track()` is given a low `conf` and the confidence filter
is applied to the results afterwards, rather than starving the tracker up front.

In [ ]:
from tqdm.auto import tqdm

trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=30)

TRACKER_INPUT_CONF = 0.1  # low on purpose: ByteTrack matches weak boxes to live tracks

frame_generator = sv.get_video_frames_generator(str(SOURCE_VIDEO_PATH))

with sv.VideoSink(str(TARGET_VIDEO_PATH), video_info) as sink:
    for frame in tqdm(frame_generator, total=video_info.total_frames):
        result = model.track(
            frame,
            conf=TRACKER_INPUT_CONF,
            persist=True,
            tracker="bytetrack.yaml",
            verbose=False,
        )[0]
        detections = sv.Detections.from_ultralytics(result)
        # Track on weak boxes, but only draw the confident ones.
        detections = detections[detections.confidence >= CONFIDENCE_THRESHOLD]

        annotated = frame.copy()
        if detections.tracker_id is not None:
            annotated = trace_annotator.annotate(scene=annotated, detections=detections)
        annotated = box_annotator.annotate(scene=annotated, detections=detections)
        annotated = label_annotator.annotate(
            scene=annotated, detections=detections, labels=make_labels(detections)
        )

        sink.write_frame(annotated)

print("wrote:", TARGET_VIDEO_PATH.resolve())

## 7. Play the result in the notebook

Supervision writes `mp4v`, which browsers will not decode. Re-encode to H.264 with
ffmpeg (preinstalled on Colab) so the video plays inline.

In [ ]:
PLAYABLE_VIDEO_PATH = Path("fire_result_h264.mp4")

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH} -vcodec libx264 -pix_fmt yuv420p {PLAYABLE_VIDEO_PATH}

print("wrote:", PLAYABLE_VIDEO_PATH.resolve())

In [ ]:
import base64

from IPython.display import HTML, display

encoded = base64.b64encode(PLAYABLE_VIDEO_PATH.read_bytes()).decode()
display(HTML(f'''
<video width="720" controls>
  <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
</video>
'''))

## 8. Download the result

In [ ]:
try:
    from google.colab import files  # type: ignore

    files.download(str(PLAYABLE_VIDEO_PATH))
except ImportError:
    print("Not running in Colab. Result is at:", PLAYABLE_VIDEO_PATH.resolve())